# 지하철 역 이용객 수에 영향을 미치는 변수에 관한 연구 (II)
## 승법적 회귀모형을 활용한 8호선 역별 주간인구밀도 추정 — 고도화

김인욱 | 가천대학교 스마트시티융합학과 석사 (202640502) | 2026.04.13.

### 주요 개선사항 (260438 → 260413)
1. **비선형 승법적 모형** 통합 수식 적용
2. **시계열적 K값 동적 할당**: 석촌(2018~), 별내/구리(2024~) 환승노선 개통 시점 반영
3. **가우시안 감쇄 상수 a 동적 추정**: 고정 2.77 → a = -ln(1-P)/d₀² = 1.331
4. **Long-format 패널 데이터** 구조 (299 관측치)
5. **잔차 기반 역추정(Inverse Estimation)** 강조 및 도시 기능 정량 분류

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import t as t_dist

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# ============================================================
# 1. 8호선 24개역 일평균 승하차량 (2008~2025)
#    환승역: 양 노선 합계 승하차량 사용
# ============================================================
data = {
    "별내": {2008:4858,2009:5218,2010:4215,2011:5033,2012:5646,2013:5896,2014:3428,2015:4150,2016:4651,2017:4889,2018:4858,2019:5218,2020:4215,2021:5033,2022:5646,2023:5896,2024:18568,2025:19523},
    "다산": {2024:16773,2025:18774},
    "동구릉": {2024:10582,2025:11735},
    "구리": {2008:20775,2009:22113,2010:24358,2011:25891,2012:26589,2013:27735,2014:28222,2015:28350,2016:28494,2017:28164,2018:28450,2019:29205,2020:22285,2021:22790,2022:24564,2023:25661,2024:37065,2025:33674},
    "장자호수공원": {2024:15742,2025:17082},
    "암사역사공원": {2024:10372,2025:11792},
    "암사": {2008:31364,2009:31530,2010:32402,2011:33067,2012:33399,2013:33815,2014:33989,2015:33301,2016:33191,2017:34058,2018:34938,2019:35535,2020:28773,2021:29525,2022:31576,2023:33348,2024:32695,2025:30241},
    "천호": {2008:82171,2009:82169,2010:86080,2011:88759,2012:89768,2013:90262,2014:90075,2015:86421,2016:83588,2017:82248,2018:82465,2019:80794,2020:60126,2021:58207,2022:62983,2023:67469,2024:71151,2025:75578},
    "강동구청": {2008:20665,2009:20277,2010:20657,2011:21248,2012:21563,2013:21826,2014:21823,2015:21586,2016:20943,2017:21403,2018:21790,2019:21274,2020:16880,2021:17232,2022:18402,2023:19757,2024:20186,2025:20443},
    "몽촌토성": {2008:12836,2009:12773,2010:13351,2011:13262,2012:13637,2013:13604,2014:13984,2015:14618,2016:13597,2017:14342,2018:14555,2019:12683,2020:9150,2021:10161,2022:11589,2023:12911,2024:14010,2025:14672},
    "잠실": {2008:165299,2009:168261,2010:167583,2011:170114,2012:169898,2013:171401,2014:177041,2015:182891,2016:192115,2017:202426,2018:207811,2019:205623,2020:136004,2021:137715,2022:166873,2023:185257,2024:193898,2025:199230},
    "석촌": {2008:18919,2009:18572,2010:19428,2011:19693,2012:19809,2013:19868,2014:19651,2015:18911,2016:18600,2017:17911,2018:26519,2019:29384,2020:24627,2021:26661,2022:30167,2023:33047,2024:35228,2025:36438},
    "송파": {2008:13885,2009:13548,2010:14044,2011:14188,2012:14120,2013:11657,2014:11346,2015:11157,2016:10837,2017:11018,2018:10895,2019:18081,2020:13674,2021:12649,2022:14847,2023:17237,2024:18052,2025:18615},
    "가락시장": {2008:23127,2009:22866,2010:28870,2011:30428,2012:30720,2013:31225,2014:31970,2015:31390,2016:33166,2017:35545,2018:36347,2019:36897,2020:28551,2021:27909,2022:30162,2023:32672,2024:33717,2025:34450},
    "문정": {2008:12293,2009:11563,2010:11357,2011:10787,2012:10506,2013:10589,2014:11000,2015:11560,2016:15301,2017:25571,2018:33715,2019:37833,2020:32577,2021:34218,2022:35930,2023:38152,2024:40313,2025:41696},
    "장지": {2008:9090,2009:11030,2010:15954,2011:19288,2012:20966,2013:22066,2014:24028,2015:25778,2016:28766,2017:31006,2018:30879,2019:31003,2020:24197,2021:25028,2022:27665,2023:29894,2024:31104,2025:31860},
    "복정": {2008:13260,2009:13475,2010:14037,2011:14410,2012:14568,2013:14705,2014:14835,2015:14727,2016:14684,2017:14638,2018:15051,2019:15329,2020:12206,2021:12576,2022:14112,2023:15494,2024:16738,2025:17322},
    "남위례": {2019:9133,2020:8771,2021:9580,2022:11461,2023:13001,2024:13746,2025:14394},
    "산성": {2008:13700,2009:13454,2010:13534,2011:13625,2012:13554,2013:13301,2014:13060,2015:12840,2016:12491,2017:12335,2018:12234,2019:12174,2020:9521,2021:9685,2022:10485,2023:11234,2024:11617,2025:12087},
    "남한산성입구": {2008:27002,2009:26735,2010:26921,2011:27010,2012:26871,2013:26498,2014:26252,2015:25753,2016:25124,2017:25053,2018:24928,2019:24692,2020:18919,2021:19287,2022:21060,2023:22610,2024:23378,2025:24179},
    "단대오거리": {2008:24455,2009:24099,2010:24286,2011:24296,2012:24072,2013:23775,2014:23365,2015:22818,2016:22225,2017:22095,2018:22056,2019:22024,2020:17264,2021:17571,2022:19311,2023:20884,2024:21767,2025:22678},
    "신흥": {2008:12027,2009:11818,2010:11805,2011:11640,2012:11463,2013:11283,2014:11024,2015:10671,2016:10310,2017:10116,2018:10054,2019:10044,2020:7862,2021:8021,2022:8758,2023:9511,2024:9990,2025:10458},
    "수진": {2008:11856,2009:11700,2010:11731,2011:11638,2012:11400,2013:11087,2014:10837,2015:10572,2016:10233,2017:10119,2018:10050,2019:9999,2020:7797,2021:8037,2022:8816,2023:9605,2024:10088,2025:10524},
    "모란": {2008:49484,2009:48724,2010:48811,2011:49073,2012:48626,2013:47876,2014:47332,2015:46483,2016:45498,2017:45149,2018:45204,2019:45335,2020:35379,2021:37037,2022:41345,2023:45336,2024:47721,2025:49378},
}

print(f"역 수: {len(data)}")
print(f"총 관측치(연도x역): {sum(len(v) for v in data.values())}")

In [ ]:
# ============================================================
# 2. 역-행정동 매핑, 면적, 수송력 상수
# ============================================================

station_dong = {
    '별내':['별내동'], '다산':['다산1동'], '동구릉':['인창동'],
    '구리':['인창동','수택1동'], '장자호수공원':['수택3동'],
    '암사역사공원':['암사제1동','암사제2동'], '암사':['암사제1동'],
    '천호':['천호제2동','성내제2동'], '강동구청':['성내제1동','풍납2동'],
    '몽촌토성':['방이2동'], '잠실':['잠실6동'],
    '석촌':['석촌동','송파1동'], '송파':['송파2동','가락본동'],
    '가락시장':['가락1동','문정2동'], '문정':['문정2동'],
    '장지':['장지동','문정2동'], '복정':['장지동','복정동'],
    '남위례':['복정동'], '산성':['산성동'],
    '남한산성입구':['단대동','은행2동'], '단대오거리':['단대동','신흥2동'],
    '신흥':['신흥3동','성남동'], '수진':['수진1동','성남동'],
    '모란':['성남동','수진2동'],
}

dong_area = {
    '별내동':18.64, '다산1동':5.55, '인창동':2.04, '수택1동':1.26,
    '수택3동':9.35, '암사제1동':1.02, '암사제2동':1.18, '천호제2동':1.57,
    '성내제2동':0.67, '성내제1동':0.58, '풍납2동':1.6, '방이2동':0.8,
    '잠실6동':2.8, '석촌동':1.0, '송파1동':0.8, '송파2동':0.5,
    '가락본동':1.1, '가락1동':1.3, '문정2동':2.2, '장지동':1.4,
    '복정동':3.55, '산성동':0.58, '단대동':0.80, '신흥2동':1.14,
    '신흥3동':0.33, '수진1동':0.33, '수진2동':0.93, '성남동':1.99,
    '은행2동':2.46,
}

K_BASE = 307_200  # 6량 x 160명/량 x 320회

# ============================================================
# [개선1] 시계열적 K값 동적 할당
#   환승노선 개통 시점 반영하여 연도별 K 배수 차등 적용
# ============================================================
K_MULT_TIMELINE = {
    '별내':    [(2008,2023,1.0), (2024,2030,1.4)],   # 2024 별내선 개통
    '구리':    [(2008,2023,1.0), (2024,2030,1.6)],   # 2024 경의중앙선
    '천호':    [(2008,2030,3.0)],                      # 5호선 기존
    '잠실':    [(2008,2030,5.0)],                      # 2호선 기존
    '석촌':    [(2008,2017,1.0), (2018,2030,2.5)],   # 2018 9호선 개통
    '가락시장': [(2008,2009,1.0), (2010,2030,2.6)],   # 2010 3호선
    '복정':    [(2008,2030,2.1)],                      # 분당선 기존
    '모란':    [(2008,2030,2.1)],                      # 분당선 기존
}

def get_K(station, year):
    timeline = K_MULT_TIMELINE.get(station)
    if timeline is None:
        return K_BASE * 1.0
    for start, end, mult in timeline:
        if start <= year <= end:
            return K_BASE * mult
    return K_BASE * 1.0

# 검증: K값 변동
print("석촌역 K값 변동 (9호선 2018년 개통 반영):")
for yr in [2016, 2017, 2018, 2019]:
    print(f"  {yr}: K = {get_K('석촌', yr):>10,.0f}  (배수 = {get_K('석촌', yr)/K_BASE:.1f})")

print()
print("별내역 K값 변동 (별내선 2024년 개통 반영):")
for yr in [2023, 2024, 2025]:
    print(f"  {yr}: K = {get_K('별내', yr):>10,.0f}  (배수 = {get_K('별내', yr)/K_BASE:.1f})")

In [ ]:
# ============================================================
# [개선2] 가우시안 감쇄 상수 a의 동적 추정
#   기존: a = 2.77 (고정값)
#   개선: a = -ln(1-P) / d0^2
# ============================================================

def compute_gaussian_a(d0, P):
    return -np.log(1 - P) / (d0 ** 2)

def interference_index(distance_km, a):
    return 1 - np.exp(-a * distance_km ** 2)

# 파라미터 설정
d0 = 1.5   # 임계 거리 (km)
P  = 0.95  # 95% 간섭 제거

a = compute_gaussian_a(d0, P)
print(f"감쇄 상수 동적 추정:")
print(f"  d0 = {d0} km, P = {P:.0%}")
print(f"  a = -ln(1 - {P}) / {d0}^2 = {a:.4f}")
print(f"  (기존 고정값 2.77 대비 차이: {abs(a-2.77):.4f})")
print()
print(f"물리적 의미: {d0}km 이상 떨어진 역은 인접 역 수요에 {P:.0%} 이상 영향을 받지 않음")

# 비교표
print()
print(f"거리별 간섭지수 비교:")
print(f"  {'거리(km)':<12} {'기존(a=2.77)':<15} {'개선(a=' + f'{a:.3f}' + ')'}")
for d in [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0]:
    old_val = interference_index(d, 2.77)
    new_val = interference_index(d, a)
    print(f"  {d:<12.1f} {old_val:<15.6f} {new_val:.6f}")

In [ ]:
# ============================================================
# 3. 인구 데이터 로드 및 Long-format 패널 데이터 구축
# ============================================================

_df_pop = pd.read_csv('101_DT_1B04005N_20260408025031.csv', encoding='euc-kr')
_df_pop.columns = ['행정구역','연령','항목','단위'] + [f'{y}' for y in range(2011,2026)] + ['_']
_df_pop = _df_pop.drop(columns=['항목','단위','_'], errors='ignore')
_df_pop = _df_pop[~_df_pop['행정구역'].isin(['강동구','송파구','수정구','중원구','구리시','남양주시'])].copy()
year_cols = [str(y) for y in range(2011,2026)]
for c in year_cols:
    _df_pop[c] = pd.to_numeric(_df_pop[c], errors='coerce').fillna(0).astype(int)

age_0_14  = ['0 - 4세','5 - 9세','10 - 14세']
age_15_64 = ['15 - 19세','20 - 24세','25 - 29세','30 - 34세','35 - 39세',
             '40 - 44세','45 - 49세','50 - 54세','55 - 59세','60 - 64세']
age_65p   = ['65 - 69세','70 - 74세','75 - 79세','80 - 84세','85 - 89세',
             '90 - 94세','95 - 99세','100+']

def get_pop(ages, dongs):
    return _df_pop[(_df_pop['행정구역'].isin(dongs)) & (_df_pop['연령'].isin(ages))][year_cols].sum()

# 역간 거리 데이터
_df_stn = pd.read_csv('수도권전철8호선_역번호_거리_역명.csv', encoding='utf-8-sig')
stn_dist = dict(zip(_df_stn['역명'], _df_stn['역세권거리']))

# 패널 데이터 구축
stations_order = list(station_dong.keys())
rows = []

for stn in stations_order:
    dongs = station_dong[stn]
    aa = sum(dong_area.get(d, 0) for d in dongs)
    p23 = get_pop(age_0_14, dongs)
    p24 = get_pop(age_15_64, dongs)
    p25 = get_pop(age_65p, dongs)
    pt  = get_pop(['계'], dongs)

    dist = stn_dist.get(stn, np.nan)
    L = interference_index(dist, a) if not np.isnan(dist) else np.nan

    for yr in range(2011, 2026):
        yv = data.get(stn, {}).get(yr)
        if yv is None or yv == 0:
            continue
        v23, v24, v25 = int(p23.get(str(yr), 0)), int(p24.get(str(yr), 0)), int(p25.get(str(yr), 0))
        vt = int(pt.get(str(yr), 0))
        if v23 == 0 and v24 == 0 and v25 == 0:
            continue

        K = get_K(stn, yr)  # [개선1] 시계열적 K값

        rows.append({
            '역명': stn, '연도': yr, 'y': yv,
            'x23': v23/aa, 'x24': v24/aa, 'x25': v25/aa,
            'nighttime_pop': vt, 'nighttime_density': vt/aa,
            'admin_area': aa,
            'K': K, 'ln_K': np.log(K),
            'L': L, 'ln_L': np.log(L) if L and L > 0 else np.nan,
        })

panel = pd.DataFrame(rows).dropna()
panel = panel[(panel['x23'] > 0) & (panel['x24'] > 0) & (panel['x25'] > 0)]
panel['ln_y'] = np.log(panel['y'])
for yr in range(2012, 2026):
    panel[f'Y_{yr}'] = (panel['연도'] == yr).astype(float)

print(f"패널 데이터 구축 완료:")
print(f"  관측치: {len(panel)}")
print(f"  역 수: {panel['역명'].nunique()}")
print(f"  연도 범위: {panel['연도'].min()} ~ {panel['연도'].max()}")
print(f"  동적 K 적용: 석촌(2018~), 별내(2024~), 구리(2024~), 가락시장(2010~)")

print()
print("석촌역 패널 데이터 (K값 변동 확인):")
print(panel[panel['역명']=='석촌'][['역명','연도','y','K','ln_K','L']].to_string(index=False))

## Pooled OLS 추정

승법적 모형의 로그 변환:

$$\ln(y_{i,t}) = c + \beta_{23}x_{23} + \beta_{24}x_{24} + \beta_{25}x_{25} + \beta_K \ln(K_{i,t}) + \beta_L \ln(L_i) + \delta_t + \epsilon_{i,t}$$

In [ ]:
# ============================================================
# 4. Pooled OLS 추정
# ============================================================

year_dummies = [f'Y_{yr}' for yr in range(2012, 2026)]
feat_ols = ['x23','x24','x25','ln_K'] + year_dummies
feat_names_ols = ['const'] + feat_ols

X_ols = np.column_stack([np.ones(len(panel)), panel[feat_ols].values])
y_ols = panel['ln_y'].values

b_ols = np.linalg.inv(X_ols.T @ X_ols) @ X_ols.T @ y_ols
r_ols = y_ols - X_ols @ b_ols
n_ols, k_ols = X_ols.shape
dof_ols = n_ols - k_ols
s2_ols = (r_ols @ r_ols) / dof_ols
se_ols = np.sqrt(np.diag(s2_ols * np.linalg.inv(X_ols.T @ X_ols)))
t_ols = b_ols / se_ols
p_ols = 2 * (1 - t_dist.cdf(np.abs(t_ols), df=dof_ols))
R2_ols = 1 - np.sum(r_ols**2) / np.sum((y_ols - y_ols.mean())**2)
R2_adj_ols = 1 - (1 - R2_ols) * (n_ols - 1) / dof_ols
DW_ols = np.sum(np.diff(r_ols)**2) / np.sum(r_ols**2)

print("=" * 70)
print("  Pooled OLS")
print("=" * 70)
print(f"  R2 = {R2_ols:.4f},  Adj.R2 = {R2_adj_ols:.4f},  N = {n_ols},  DW = {DW_ols:.3f}")
print()
print(f"  {'변수':<10} {'계수':>12} {'SE':>12} {'t':>8} {'p':>10}")
print("  " + "-" * 60)
for v in ['x23','x24','x25','ln_K']:
    i = feat_names_ols.index(v)
    sig = '***' if p_ols[i]<0.01 else '**' if p_ols[i]<0.05 else '*' if p_ols[i]<0.1 else ''
    print(f"  {v:<10} {b_ols[i]:>+12.6f} {se_ols[i]:>12.6f} {t_ols[i]:>8.3f} {p_ols[i]:>10.4f} {sig}")

bk = b_ols[feat_names_ols.index('ln_K')]
print(f"\n  [수송력 탄력성] beta_K = {bk:.3f}")
print(f"    -> 수송 용량 10% 증가 시 이용객 약 {bk*10:.1f}% 증가")
if bk > 1:
    print(f"    -> beta_K > 1: 공급 확대의 수요 유인 효과가 큰 상태 (수요 탄력적)")

## 역 고정효과(FE) 모형 - Within Estimator

FE 모형은 각 역의 시불변 특성을 역별 절편으로 흡수하고, 시간 내 변동만으로 계수를 추정한다.

**개선**: ln(K)가 시계열적으로 변동하므로 FE에서도 수송력 탄력성 추정이 가능하다.

In [ ]:
# ============================================================
# 5. 역 고정효과(FE) - Within Estimator
# ============================================================

time_varying = ['ln_y','x23','x24','x25','ln_K'] + year_dummies

dm = panel.copy()
for v in time_varying:
    dm[v+'_dm'] = dm.groupby('역명')[v].transform(lambda x: x - x.mean())

fe_feat = ['x23_dm','x24_dm','x25_dm','ln_K_dm'] + [f'{d}_dm' for d in year_dummies]
fe_names = ['x23','x24','x25','ln_K'] + year_dummies

X_fe = dm[fe_feat].values
y_fe = dm['ln_y_dm'].values

n_stn = panel['역명'].nunique()
n_fe = len(y_fe)
k_fe_n = X_fe.shape[1]
dof_fe = n_fe - n_stn - k_fe_n

b_fe = np.linalg.inv(X_fe.T @ X_fe) @ X_fe.T @ y_fe
r_fe_dm = y_fe - X_fe @ b_fe
s2_fe = (r_fe_dm @ r_fe_dm) / dof_fe
se_fe = np.sqrt(np.diag(s2_fe * np.linalg.inv(X_fe.T @ X_fe)))
t_fe = b_fe / se_fe
p_fe = 2 * (1 - t_dist.cdf(np.abs(t_fe), df=dof_fe))
DW_fe = np.sum(np.diff(r_fe_dm)**2) / np.sum(r_fe_dm**2)

# alpha_i 복원
alpha_i = {}
for stn in panel['역명'].unique():
    m = panel['역명'] == stn
    alpha_i[stn] = panel.loc[m,'ln_y'].mean() - panel.loc[m, fe_names].mean().values @ b_fe

panel['alpha_i'] = panel['역명'].map(alpha_i)
panel['ln_y_hat_fe'] = panel['alpha_i']
for j, v in enumerate(fe_names):
    panel['ln_y_hat_fe'] += b_fe[j] * panel[v]
r_fe_full = panel['ln_y'].values - panel['ln_y_hat_fe'].values
R2_fe = 1 - np.sum(r_fe_full**2) / np.sum((y_ols - y_ols.mean())**2)

print("=" * 70)
print("  역 고정효과(FE)")
print("=" * 70)
print(f"  R2 = {R2_fe:.4f},  N = {n_fe},  DW = {DW_fe:.3f},  역 고정효과: {n_stn}개")
print()
print(f"  {'변수':<10} {'계수':>12} {'SE':>12} {'t':>8} {'p':>10}")
print("  " + "-" * 60)
for v in ['x23','x24','x25','ln_K']:
    i = fe_names.index(v)
    sig = '***' if p_fe[i]<0.01 else '**' if p_fe[i]<0.05 else '*' if p_fe[i]<0.1 else ''
    print(f"  {v:<10} {b_fe[i]:>+12.6f} {se_fe[i]:>12.6f} {t_fe[i]:>8.3f} {p_fe[i]:>10.4f} {sig}")

## Pooled OLS vs FE 비교

In [ ]:
# ============================================================
# 6. Pooled OLS vs FE 비교
# ============================================================

print("=" * 80)
print("  Pooled OLS vs FE 비교")
print("=" * 80)
print(f"\n{'':>25} {'Pooled OLS':>15} {'FE':>15}")
print("-" * 60)
print(f"{'R2':>25} {R2_ols:>15.4f} {R2_fe:>15.4f}")
print(f"{'Adj.R2':>25} {R2_adj_ols:>15.4f} {'':>15}")
print(f"{'Durbin-Watson':>25} {DW_ols:>15.3f} {DW_fe:>15.3f}")
print(f"{'N':>25} {n_ols:>15} {n_fe:>15}")

print(f"\n{'변수':<12} {'OLS':>12} {'FE':>12} {'해석'}")
print("-" * 75)
interp = {
    'x23': 'FE: 유소년 감소 -> 이용객 증가',
    'x24': 'FE: 생산인구 증가 -> 이용객 증가 (부호 전환!)',
    'x25': 'FE: 고령화 -> 이용객 감소',
    'ln_K': '수송력 탄력성 (동적 K로 FE에서도 추정)',
}
for v in ['x23','x24','x25','ln_K']:
    oi = feat_names_ols.index(v)
    if v in fe_names:
        fi = fe_names.index(v)
        print(f"  {v:<10} {b_ols[oi]:>+12.6f} {b_fe[fi]:>+12.6f}  {interp.get(v,'')}")


# 연도 고정효과
print(f"\n{'연도':<8} {'OLS':>10} {'FE':>10} {'FE p':>10}")
print("-" * 42)
for yr in range(2012, 2026):
    v = f'Y_{yr}'
    oi = feat_names_ols.index(v)
    fi = fe_names.index(v)
    sig = '***' if p_fe[fi]<0.01 else '**' if p_fe[fi]<0.05 else '*' if p_fe[fi]<0.1 else ''
    print(f"  {yr}  {b_ols[oi]:>+10.4f} {b_fe[fi]:>+10.4f} {p_fe[fi]:>10.4f} {sig}")

## 잔차 기반 주간인구밀도 역추정 (Inverse Estimation)

$$\hat{x}_{day,i,t} = x_{night,i,t} \times e^{\hat{\epsilon}_{i,t}}$$

- $e^{\hat{\epsilon}} > 1$: 주간 유입 (업무/상업)
- $e^{\hat{\epsilon}} < 1$: 주간 유출 (베드타운)

In [ ]:
# ============================================================
# 7. 주간인구밀도 역추정
# ============================================================

panel['ratio_ols'] = np.exp(r_ols)
panel['day_density_ols'] = panel['nighttime_density'] * panel['ratio_ols']
panel['ratio_fe'] = np.exp(r_fe_full)
panel['day_density_fe'] = panel['nighttime_density'] * panel['ratio_fe']

target = panel[panel['연도'] == 2025].copy()
ols_sorted = target.sort_values('ratio_ols', ascending=False)

def classify(ratio):
    if ratio >= 2.0: return "강한 업무/상업"
    elif ratio >= 1.3: return "업무/상업 유입"
    elif ratio >= 0.9: return "균형"
    elif ratio >= 0.6: return "주거 우세"
    else: return "베드타운"

# OLS 기반 결과
print("=" * 100)
print("  주간인구밀도 역추정 (OLS, 2025, e^epsilon 내림차순)")
print("=" * 100)
print(f"  {'순위':>4} {'역명':<14} {'야간밀도':>10} {'이용객':>8} {'e^eps':>8} {'주간밀도':>10} {'도시기능'}")
print("  " + "-" * 85)

for rank, (_, r) in enumerate(ols_sorted.iterrows(), 1):
    func = classify(r['ratio_ols'])
    marker = ">>>" if r['ratio_ols'] > 1.3 else "<<<" if r['ratio_ols'] < 0.7 else "   "
    print(f"  {rank:>4} {r['역명']:<14} {r['nighttime_density']:>10,.0f} {r['y']:>8,} "
          f"{r['ratio_ols']:>8.2f} {r['day_density_ols']:>10,.0f} {marker} {func}")

In [ ]:
# ============================================================
# 8. OLS vs FE 주간인구밀도 비교
# ============================================================

print("=" * 115)
print("  OLS vs FE 주간인구밀도 비교 (2025)")
print("=" * 115)
print(f"  {'순위':>4} {'역명':<14} {'alpha_i':>8} {'야간밀도':>10} {'이용객':>8} {'OLS비율':>8} {'FE비율':>8} {'OLS주간밀도':>12} {'FE주간밀도':>12}")
print("  " + "-" * 105)

compared = target.sort_values('ratio_ols', ascending=False)
for rank, (_, r) in enumerate(compared.iterrows(), 1):
    print(f"  {rank:>4} {r['역명']:<14} {alpha_i[r['역명']]:>8.3f} {r['nighttime_density']:>10,.0f} "
          f"{r['y']:>8,} {r['ratio_ols']:>8.2f} {r['ratio_fe']:>8.2f} "
          f"{r['day_density_ols']:>12,.0f} {r['day_density_fe']:>12,.0f}")

print()
print("  [해석]")
print("  OLS: 역 간 차이를 포착 -> 주간인구 역추정에 적합 (업무/주거 식별)")
print("  FE:  역 고정효과 통제 -> 계수 해석 및 최근 성장 추세 식별에 적합")

## 도시 기능 정량적 분류 및 검증

In [ ]:
# ============================================================
# 9. 도시 기능 분류 요약 및 검증
# ============================================================

print("=" * 80)
print("  도시 기능 분류 (OLS e^epsilon 기준)")
print("=" * 80)

cats = {
    "강한 업무/상업 (e^eps >= 2.0)": [],
    "업무/상업 유입 (1.3 <= e^eps < 2.0)": [],
    "균형 (0.9 <= e^eps < 1.3)": [],
    "주거 우세 (0.6 <= e^eps < 0.9)": [],
    "베드타운 (e^eps < 0.6)": [],
}

for _, r in ols_sorted.iterrows():
    func = classify(r['ratio_ols'])
    entry = f"{r['역명']}({r['ratio_ols']:.2f})"
    for cn, cl in cats.items():
        if func in cn:
            cl.append(entry)
            break
    else:
        cats["강한 업무/상업 (e^eps >= 2.0)"].append(entry)

for cat, items in cats.items():
    if items:
        print(f"  {cat}:")
        print(f"    {', '.join(items)}")
        print()

# 검증
print("-" * 80)
print("  추정 결과 vs 실제 도시 기능 부합도")
print("-" * 80)
checks = [
    ("문정", "법조단지, 가든파이브", True),
    ("장지", "가든파이브, 문정현대지식산업센터", True),
    ("잠실", "롯데월드타워, COEX권", True),
    ("산성", "재개발 진행, 주거지", False),
    ("수진", "주거 밀집, 출퇴근 유출", False),
    ("다산", "신도시 주거단지", False),
]
for stn, desc, is_biz in checks:
    row = target[target['역명'] == stn]
    if len(row) > 0:
        rv = row.iloc[0]['ratio_ols']
        ok = (is_biz and rv > 1.3) or (not is_biz and rv < 0.9)
        status = "[O] 부합" if ok else "[?] 검토 필요"
        print(f"  {stn:<10} e^eps = {rv:.2f}  {status}  ({desc})")

## 역세권 행정동 기본 정보 (2025년)

In [ ]:
# ============================================================
# 10. 역세권 행정동 기본 정보
# ============================================================

print(f"  {'역명':<14} {'행정동':<35} {'면적(km2)':>10} {'야간인구':>10} {'야간밀도':>10}")
print("  " + "-" * 85)

for stn in stations_order:
    dongs = station_dong[stn]
    aa = sum(dong_area.get(d, 0) for d in dongs)
    row = target[target['역명'] == stn]
    if len(row) > 0:
        r = row.iloc[0]
        dong_str = ', '.join(dongs)
        print(f"  {stn:<14} {dong_str:<35} {aa:>10.2f} {r['nighttime_pop']:>10,.0f} {r['nighttime_density']:>10,.0f}")